# Assemble outputs from existing results

Read-only finisher. The simulations (headline grid, trajectory, polynomial
baselines, cross-fit SR-PDS) have already been run and saved. This notebook
**re-runs none of them**. It only:

1. builds the comprehensive recovery table (the one thing not yet computed),
2. redraws the trajectory figure from saved results,
3. assembles the fixed-n metrics table (headline + improved CF + poly-2).

The only computation is the polynomial recovery rates (LASSO only, a few
minutes). Run top to bottom; each cell is self-contained.

**After a kernel restart, just run the cells in order — no warm-up or CF run.**


## What's on disk


In [ ]:
import os
import config as C
print('results dir:', C.RESULTS_DIR.resolve(), '\n')
for f in sorted(os.listdir(C.RESULTS_DIR)):
    if f.endswith(('.pkl', '.csv')):
        print('  ', f)


## 1. Comprehensive recovery table

Per (DGP, equation, true term): **pre-LASSO** rate (term got into the
dictionary) and **post-LASSO** rate (term kept as a control), for SR-PDS and
both polynomial baselines. SR-PDS recovery is stochastic (the PySR search); the
polynomial rates are deterministic (a term is a basis column or not). The two
failure modes are visible: poly-2 cannot represent the cubic (0%), and no
polynomial basis represents the log term at any degree, whereas SR-PDS with a
log operator can. Writes `recovery_full.csv`.


In [ ]:
import numpy as np, pandas as pd, pickle
import config as C
from dgp import DGP_REGISTRY
from evaluate import _term_label
from srpds_poly import poly_selection

RD = C.RESULTS_DIR
NREP_POLY = 100        # cheap: LASSO only

# 1. SR-PDS (same-data) pre/post from recovery.csv or a saved log
sr = None
if (RD/'recovery.csv').exists():
    r = pd.read_csv(RD/'recovery.csv'); r = r[r['variant'] == 'sr_pds']
    sr = (r.pivot_table(index=['dgp','equation','term','term_label'],
                        columns='stage', values='recovery_rate').reset_index())
    sr['method'] = 'SR-PDS'
else:
    for lp in ('sr_pds_log_headline.pkl', 'sr_pds_log.pkl'):
        if (RD/lp).exists():
            from evaluate import evaluate_recovery
            rr = evaluate_recovery(pickle.load(open(RD/lp,'rb')), DGP_REGISTRY)
            rr = rr[rr['variant'] == 'sr_pds']
            sr = (rr.pivot_table(index=['dgp','equation','term','term_label'],
                                 columns='stage', values='recovery_rate').reset_index())
            sr['method'] = 'SR-PDS'; break
if sr is None:
    print('WARNING: no SR-PDS recovery source found -- polynomial baselines only')

# 2. polynomial pre(representable) / post(selected), both equations
DEG = {2: 'PDS-LASSO (poly-2)', 3: 'PDS-LASSO (poly-3)'}
poly_rows = []
for dgp_key, e in DGP_REGISTRY.items():
    ty, td = e.get('truth_terms_y', []), e.get('truth_terms_d', [])
    allt = list(dict.fromkeys(list(ty) + list(td)))
    if not allt:
        continue
    for deg, mlabel in DEG.items():
        acc = {t: {'r': [], 's': []} for t in allt}
        for seed in range(NREP_POLY):
            X, d, y, _ = e['fn'](n=C.N_HEADLINE, p=C.P, s=C.S, beta0=C.BETA0, seed=seed)
            sel = poly_selection(X, d, y, C.N_HEADLINE, deg, allt)
            for t in allt:
                acc[t]['r'].append(sel[t]['representable'])
                acc[t]['s'].append(sel[t]['selected'])
        for t in allt:
            pre, post = float(np.mean(acc[t]['r'])), float(np.mean(acc[t]['s']))
            for eq, terms in (('y', ty), ('d', td)):
                if t in terms:
                    poly_rows.append({'dgp': dgp_key, 'equation': eq, 'term': str(t),
                                      'term_label': _term_label(t), 'method': mlabel,
                                      'pre_lasso': round(pre,3), 'post_lasso': round(post,3)})
    print(f'  poly recovery {dgp_key} done', flush=True)
poly = pd.DataFrame(poly_rows)

# 3. merge, save, display
cols = ['dgp','equation','term','term_label','method','pre_lasso','post_lasso']
frames = [poly]
if sr is not None:
    frames.insert(0, sr[cols])
recovery_full = pd.concat(frames, ignore_index=True)
recovery_full['dgp_label'] = recovery_full['dgp'].map(lambda k: DGP_REGISTRY[k]['label'])
recovery_full = recovery_full.sort_values(['dgp','equation','term_label','method'])
recovery_full.to_csv(RD/'recovery_full.csv', index=False)
print('\nwrote recovery_full.csv')

wide = recovery_full.pivot_table(index=['dgp_label','equation','term_label'],
                                 columns='method', values=['pre_lasso','post_lasso'])
wide


## 2. Trajectory figure (read-only)

Figure 1: bias and RMSE vs n on the three confounding designs, all methods plus
the polynomial baselines, SR-PDS in black. Reads saved trajectory results and
poly files; capped at 5,000. Shows inline and saves to `figures/`.


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FixedFormatter, NullLocator
import config as C
RD, NMAX = C.RESULTS_DIR, 5000

def _agg_og():
    frames = [pd.read_pickle(RD/f) for f in ('trajectory.pkl','large_n.pkl') if (RD/f).exists()]
    if not frames: return pd.DataFrame(columns=['dgp','est_label','n_grid','bias','rmse'])
    d = pd.concat(frames, ignore_index=True); d = d[~d['failed'].astype(bool)].copy()
    d['err'] = d['beta_hat'] - d['beta0']
    return (d.groupby(['dgp','est_label','n_grid'], as_index=False)
              .agg(bias=('err','mean'), rmse=('err', lambda e: np.sqrt(np.mean(e**2)))))

def _agg_poly(stem, label):
    pkl, csv = RD/f'{stem}.pkl', RD/f'{stem}.csv'
    if pkl.exists():
        d = pd.read_pickle(pkl); d = d[~d['failed'].astype(bool)].copy()
        d['err'] = d['beta_hat'] - d['beta0']
        a = (d.groupby(['dgp','n_grid'], as_index=False)
               .agg(bias=('err','mean'), rmse=('err', lambda e: np.sqrt(np.mean(e**2)))))
    elif csv.exists():
        a = pd.read_csv(csv).rename(columns={'n':'n_grid'})[['dgp','n_grid','bias','rmse']].copy()
    else:
        return None
    a['est_label'] = label; return a

agg = _agg_og()
for stem, lab in [('pds_poly2_trajectory','PDS-LASSO (poly-2)'),
                  ('pds_poly3_trajectory','PDS-LASSO (poly-3)')]:
    a = _agg_poly(stem, lab)
    if a is not None: agg = pd.concat([agg, a], ignore_index=True)
agg = agg[agg['n_grid'] <= NMAX]
print('methods on figure:', sorted(agg['est_label'].unique()))

KEEP = {'dgp8':('Weak NL (dummies)',0.80),'dgp6':('Mild NL conf.',1.20),'dgp7':('Severe NL conf.',1.77)}
COLORS = {'Full OLS':'#9e9e9e','PDS-LASSO':'#bcbd22','DML-LASSO':'#1f77b4','DML-RF':'#9467bd',
          'DML-NN':'#2ca02c','SR-PDS':'#000000','PDS-LASSO (poly-2)':'#d62728','PDS-LASSO (poly-3)':'#ff7f0e'}
NTICKS = [n for n in [30,100,300,1000,5000,10000] if n <= NMAX]
is_sr = lambda l: 'sr' in l.lower() and 'pds' in l.lower()
methods = sorted(agg['est_label'].unique(), key=lambda l: (is_sr(l), l))
cols = [c for c in KEEP if c in set(agg['dgp'])]

with plt.rc_context({'font.size':8.5,'axes.titlesize':9,'axes.labelsize':8.5,
                     'legend.fontsize':7.5,'xtick.labelsize':7.5,'ytick.labelsize':7.5}):
    fig, axes = plt.subplots(2, len(cols), figsize=(7.6,4.0), squeeze=False, constrained_layout=True)
    for c, code_ in enumerate(cols):
        title, eb = KEEP[code_]; sub = agg[agg['dgp']==code_]
        ax_b, ax_r = axes[0,c], axes[1,c]
        for m in methods:
            dm = sub[sub['est_label']==m].sort_values('n_grid')
            if dm.empty: continue
            hero = is_sr(m)
            st = dict(color=COLORS.get(m,'#777'), lw=2.0 if hero else 1.1,
                      marker='o', ms=3 if hero else 2.2, zorder=5 if hero else 2)
            ax_b.plot(dm['n_grid'], dm['bias'], label=m, **st)
            ax_r.plot(dm['n_grid'], dm['rmse'], label=m, **st)
        ax_b.axhline(0.0, ls=':', lw=0.7, color='0.5')
        if eb is not None: ax_b.axhline(eb, ls=':', lw=0.8, color='0.6')
        ax_b.set_title(title)
        for ax in (ax_b, ax_r):
            ax.set_xscale('log'); ax.set_xlim(right=NMAX*1.15)
            ax.xaxis.set_major_locator(FixedLocator(NTICKS))
            ax.xaxis.set_major_formatter(FixedFormatter([f'{v:,}' for v in NTICKS]))
            ax.xaxis.set_minor_locator(NullLocator())
            ax.grid(True, which='major', ls='-', lw=0.3, color='0.9'); ax.tick_params(length=2)
        ax_b.tick_params(labelbottom=False)
        ax_r.set_yscale('log'); ax_r.set_xlabel('sample size $n$')
    axes[0,0].set_ylabel('Bias', fontweight='bold')
    axes[1,0].set_ylabel('RMSE (log scale)', fontweight='bold')
    handles, lbls = axes[0,0].get_legend_handles_labels()
    fig.legend(handles, lbls, loc='outside lower center', ncol=4, frameon=False,
               handlelength=1.6, columnspacing=1.2)
    C.FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(C.FIG_DIR/'n_trajectory_confounding.png', dpi=300, bbox_inches='tight')
    plt.show()
print('saved ->', C.FIG_DIR/'n_trajectory_confounding.png')


## 3. Assembled fixed-n tables (read-only)

Folds the improved CF coverage and the poly-2 column into the headline metrics.
Reads existing files only. Writes `all_metrics_final.csv`.


In [ ]:
import numpy as np, pandas as pd
import config as C
from dgp import DGP_REGISTRY
from evaluate import evaluate
RD = C.RESULTS_DIR

frames = []
if (RD/'headline_metrics.csv').exists():
    base = pd.read_csv(RD/'headline_metrics.csv')
    base = base[base['estimator'] != 'sr_pds_cf']      # drop old CF, replace below
    frames.append(base)

if (RD/'cf_improved.pkl').exists():
    cf = pd.read_pickle(RD/'cf_improved.pkl')
    for k, g in cf.groupby('dgp'):
        m = evaluate(g, C.BETA0)
        m.update({'dgp':k,'estimator':'sr_pds_cf','est_label':'SR-PDS (CF)',
                  'dgp_label':DGP_REGISTRY[k]['label']})
        frames.append(pd.DataFrame([m]))

if (RD/'pds_poly2_fixedn.csv').exists():
    frames.append(pd.read_csv(RD/'pds_poly2_fixedn.csv'))

if frames:
    combined = pd.concat(frames, ignore_index=True)
    combined.to_csv(RD/'all_metrics_final.csv', index=False)
    print('wrote all_metrics_final.csv\n')
    for metric in ('bias','coverage'):
        print(metric.upper())
        print(combined.pivot_table(index='dgp', columns='est_label', values=metric).round(3).to_string())
        print()
else:
    print('no metric files found to assemble')


## Done

Outputs: `recovery_full.csv` (comprehensive recovery), `figures/n_trajectory_confounding.png` (Figure 1), `all_metrics_final.csv` (Tables 3-5). Nothing expensive was re-run.
